[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/25_flash_attention_solution.ipynb)

# ✅ Solution: flash_attention

Implement **tiled attention with online softmax** — the core idea behind Flash Attention.

### Signature
```python
def flash_attention(q_BLK, k_BMK, v_BMK, block_size=32) -> Tensor:
    # Q, K, V: (B, S, D)
    # Returns: (B, S, D) — same as standard attention
```

### Key Insight
Instead of materializing the full S×S attention matrix, process in blocks:
1. For each Q-block, iterate over K/V blocks
2. Use **online softmax**: track running `max` and `sum`
3. Rescale accumulator when max changes: `acc *= exp(old_max - new_max)`
4. Final: `output = acc / row_sum`

Must give **identical** results to standard softmax attention.


In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge')
except ImportError:
    pass


In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx
import math


In [ ]:
# ✅ SOLUTION

import jax, jax.numpy as jnp, math
def flash_attention(q_BLK, k_BMK, v_BMK, block_size=32):
    chunks = []
    for i in range(0, q_BLK.shape[1], block_size):
        q_block = q_BLK[:, i : i + block_size]
        m = jnp.full(q_block.shape[:-1] + (1,), -jnp.inf)
        l = jnp.zeros(q_block.shape[:-1] + (1,))
        acc = jnp.zeros(q_block.shape[:-1] + (v_BMK.shape[-1],))
        for j in range(0, k_BMK.shape[1], block_size):
            k_block = k_BMK[:, j : j + block_size]
            v_block = v_BMK[:, j : j + block_size]
            score = q_block @ jnp.swapaxes(k_block, -2, -1) / math.sqrt(q_BLK.shape[-1])
            m_new = jnp.maximum(m, jnp.max(score, -1, keepdims=True))
            e = jnp.exp(score - m_new)
            corr = jnp.exp(m - m_new)
            acc = acc * corr + e @ v_block
            l = l * corr + jnp.sum(e, -1, keepdims=True)
            m = m_new
        chunks.append(acc / l)
    return jnp.concatenate(chunks, axis=1)


In [ ]:
# Verify
print(flash_attention)


In [ ]:
from jax_judge import check
check("flash_attention")
